In [1]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-sonnet-4-5"

In [2]:
# Helper functions
from anthropic.types import Message

# Magic string to trigger redacted thinking
thinking_test_str = "ANTHROPIC_MAGIC_STRING_TRIGGER_REDACTED_THINKING_46C9A13E193C177646C7398A98432ECCCE4C1253D5E2D82641AC0E52CC2876CB"


def add_user_message(messages, message):
    user_message = {
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(user_message)


def add_assistant_message(messages, message):
    assistant_message = {
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(assistant_message)


def chat(
    messages,
    system=None,
    temperature=1.0,
    stop_sequences=[],
    tools=None,
    thinking=False,
    thinking_budget=1024,
):
    params = {
        "model": model,
        "max_tokens": 4000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if thinking:
        params["thinking"] = {
            "type": "enabled",
            "budget_tokens": thinking_budget,
            
        }

    if tools:
        params["tools"] = tools

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message


def text_from_message(message):
    return "\n".join([block.text for block in message.content if block.type == "text"])

In [3]:
prompt = """
Analyze the attached satellite image of a property with these specific steps:

1. Residence identification: Locate the primary residence on the property by looking for:
    - The largest roofed structure
    - Typical residential features (driveway connection, regular geometry)
    - Distinction from other structures (garages, sheds, pools)
    Describe the residence's location relative to property boundaries and other features.
2. Tree overhang analysis: Examine all trees near the primary residence:
    - Identify any trees whose canopy extends directly over any portion of the roof
    - Estimate the percentage of roof covered by overhanging branches (0-25%, 25-50%, 50-75%, 75-100%)
    - Note particularly dense areas of overhang

3. Fire risk assessment: For any overhanging trees, evaluate:
    - Potential wildfire vulnerability (ember catch points, continuous fuel paths to structure)
    - Proximity to chimneys, vents, or other roof openings if visible
    - Areas where branches create a "bridge" between wildland vegetation and the structure

4. Defensible space identification: Assess the property's overall vegetative structure:
    - Identify if trees connect to form a continuous canopy over or near the home
    - Note any obvious fuel ladders (vegetation that can carry fire from ground to tree to roof)

5. Fire risk rating: Based on your analysis, assign a Fire Risk Rating from 1-4:
    - Rating 1 (Low Risk): No tree branches overhanging the roof, good defensible space around the structure
    - Rating 2 (Moderate Risk): Minimal overhang (<25% of roof), some separation between tree canopies and the structure
    - Rating 3 (High Risk): Significant overhang (25-50% of roof), connected tree canopies, multiple fuel ladders
    - Rating 4 (Severe Risk): Extensive overhang (>50% of roof, dense vegetation against the structure

For each item above (1-5), write one sentence summarizing your findings, with your final response being the numeric Fire Risk Rating (1-4) with a brief justification.
"""

In [4]:
# Send each property image to Claude along with the prompt
import base64


def assess_property(image_path):
    with open(image_path, "rb") as f:
        image_bytes = base64.standard_b64encode(f.read()).decode("utf-8")

    messages = []
    add_user_message(messages, [
        {
            "type": "image",
            "source": {
                "type": "base64",
                "media_type": "image/png",
                "data": image_bytes,
            },
        },
        {"type": "text", "text": prompt},
    ])

    return text_from_message(chat(messages))


for i in range(1, 8):
    print(f"=== prop{i}.png ===")
    print(assess_property(f"images/prop{i}.png"), "\n")

=== prop1.png ===


# Satellite Image Fire Risk Analysis

## 1. Residence Identification
The primary residence is a gray-roofed structure located in the center-left portion of the property, distinguishable by its rectangular shape and what appears to be a driveway or access path connecting to it from the lower portion of the image.

## 2. Tree Overhang Analysis
Multiple large tree canopies directly overhang significant portions of the residence's roof, with dense dark green foliage covering approximately 50-75% of the roofline, particularly concentrated on the northern and eastern sides of the structure.

## 3. Fire Risk Assessment
The overhanging trees create direct contact points between vegetation and the roof surface, establishing continuous fuel pathways that would allow fire to easily transition from wildland vegetation to the structure, with particularly concerning dense canopy coverage that could trap embers and ignite the roof.

## 4. Defensible Space Identification
The property shows minimal def

# Property Fire Risk Analysis

## 1. Residence Identification
The primary residence is a large rectangular structure with a light gray/tan roof located in the center-bottom portion of the property, distinguished by its size and connection to a driveway on the left side, with a swimming pool visible behind (north of) the house.

## 2. Tree Overhang Analysis
Multiple large trees with substantial canopies are positioned immediately adjacent to the residence, with significant overhanging branches extending over approximately 40-50% of the roof area, particularly concentrated on the northern and western portions of the structure.

## 3. Fire Risk Assessment
The overhanging trees create direct fuel bridges to the roof structure, with dense canopy coverage providing multiple potential ember catch points and continuous fuel pathways that could facilitate fire spread from surrounding vegetation directly to the residence.

## 4. Defensible Space Identification
The property exhibits connected tre

# Satellite Image Fire Risk Analysis

## 1. Residence Identification
The primary residence is the main house structure with a dark (appears black/dark green) roof located in the center-upper portion of the property, positioned adjacent to what appears to be a white/light-colored patio or deck area and a kidney-shaped swimming pool to its south.

## 2. Tree Overhang Analysis
There is minimal to no visible tree canopy directly overhanging the primary residence roof, with trees positioned primarily around the perimeter of the property and maintaining clear separation from the main structure; overhang coverage appears to be 0-5% at most.

## 3. Fire Risk Assessment
The residence shows good separation from surrounding vegetation with no apparent continuous fuel pathways from trees to the structure, and no visible "bridges" of vegetation that would allow fire to easily transfer from wildland areas to the roof.

## 4. Defensible Space Identification
The property demonstrates adequate defensib

# Property Fire Risk Analysis

## 1. Residence Identification
The primary residence is a large single-family home located in the center of the property, featuring a complex multi-gabled roof structure with gray/dark roofing material and a light-colored driveway extending from the bottom right corner of the image.

## 2. Tree Overhang Analysis
There are multiple mature trees surrounding the residence, with at least one prominent tree in the upper left quadrant whose canopy appears to extend directly over or very close to the northern/northwestern portion of the roof, representing approximately 10-20% coverage of the total roof area.

## 3. Fire Risk Assessment
The overhanging tree in the northwest creates a potential ember catch point and direct fuel-to-structure pathway, particularly concerning as it appears to be positioned near roof valleys where debris could accumulate, though the overall density of overhang appears moderate rather than severe.

## 4. Defensible Space Identification

# Property Fire Risk Analysis

**1. Residence Identification:**
The primary residence is located in the upper-central portion of the property, identifiable as a dark-roofed structure with a connected driveway and what appears to be an attached or adjacent vehicle parked nearby, distinguishable from the smaller outbuilding structure visible in the lower-middle section of the property.

**2. Tree Overhang Analysis:**
Multiple large, dense tree canopies directly overhang substantial portions of the primary residence's roof, with an estimated 50-75% of the roof area covered by overhanging branches from at least 3-4 mature trees creating continuous shade over the structure.

**3. Fire Risk Assessment:**
The overhanging trees create multiple ember catch points across the majority of the roof surface, establish direct fuel pathways from the surrounding vegetation to the structure, and form a continuous canopy that bridges wildland vegetation directly to the home with minimal defensible space.

# Property Fire Risk Analysis

## 1. Residence Identification
The primary residence is a large single-story structure with a distinctive hip roof located in the center of the property, connected to a driveway from the north, and is clearly distinguishable from smaller outbuildings visible to the northwest.

## 2. Tree Overhang Analysis
Multiple large tree canopies directly overhang substantial portions of the primary residence roof, with estimated coverage of approximately 50-75% of the total roof area, particularly dense on the northern, eastern, and southern sections of the structure.

## 3. Fire Risk Assessment
The overhanging trees create multiple ember catch points across the roof surface and establish direct fuel paths from the surrounding vegetation to the structure, with particularly concerning continuous canopy coverage that could channel fire directly onto the building.

## 4. Defensible Space Identification
The property shows extensively connected tree canopies forming a nea

# Property Fire Risk Analysis

## 1. Residence Identification
The primary residence is a light-colored (likely gray or tan) roofed structure located in the center of the image, with an irregular L-shaped or cross-shaped footprint, clearly distinguishable as the main dwelling and positioned within a heavily wooded area surrounded by dense tree canopy on all sides.

## 2. Tree Overhang Analysis
Multiple large trees have canopies that extend directly over approximately 40-60% of the residence's roof area, with particularly dense overhang visible on the eastern and northern portions of the structure, creating substantial coverage across multiple roof sections.

## 3. Fire Risk Assessment
The overhanging trees create significant wildfire vulnerability by providing direct pathways for embers to accumulate on the roof and establish continuous fuel connections between the surrounding forest canopy and the structure itself, with minimal visible clearance between tree crowns and roofing material

In [5]:
import base64
with open("images/earth.pdf", "rb") as f:
    file_bytes = base64.standard_b64encode(f.read()).decode("utf-8")

messages = []
add_user_message(messages,[
    {
        "type":"document",
        "source":{
            "type": "base64",
            "media_type": "application/pdf",
            "data": file_bytes,
        },
        "title":"earth.pdf",
        "citations": {"enabled": True},
    },
    {
        "type": "text",
        "text": "How were Earth's atmosphere and oceans were formed?"},
])

chat(messages)


Message(id='msg_011CfVjxsvmtDAf5gbTvCBoH', container=None, content=[TextBlock(citations=[CitationPageLocation(cited_text="[42]\r\nEarth's atmosphere and oceans were formed by volcanic activity and outgassing.\r\n", document_index=0, document_title='earth.pdf', end_page_number=5, file_id=None, start_page_number=4, type='page_location')], text="Earth's atmosphere and oceans were formed by volcanic activity and outgassing.", type='text'), TextBlock(citations=None, text=' ', type='text'), TextBlock(citations=[CitationPageLocation(cited_text='[43] Water vapor from\r\nthese sources condensed into the oceans, augmented by water and ice from asteroids, protoplanets,\r\nand comets.\r\n', document_index=0, document_title='earth.pdf', end_page_number=5, file_id=None, start_page_number=4, type='page_location')], text='Water vapor from these sources condensed into the oceans, augmented by water and ice from asteroids, protoplanets, and comets.', type='text')], model='claude-sonnet-4-5-20250929', ro